# xbrl-frontier Phase 0 on a Kaggle T4 (owner-run)

Runs, in order: the base-model throughput benchmark (`serving.phase0_throughput_benchmark`), base-model quantization for the smoke test, and the T4 vLLM smoke test (`serving.smoke_test`). All numbers come from `docs/config/thresholds.yaml` inside the attached repo snapshot; nothing is set in this notebook.

**Settings:** Accelerator = GPU T4 (one GPU is used), Internet = on, attach the Kaggle dataset `xbrl-frontier-src` (the `git archive` tarball, see docs/OPEN.md).

**Download afterwards:** `/kaggle/working/phase0_kaggle_t4.tar.gz`, `/kaggle/working/smoke_kaggle_t4.tar.gz`, `/kaggle/working/smoke_checkpoints.tar.gz`.

In [ ]:
import glob, os, subprocess, sys
src = glob.glob('/kaggle/input/xbrl-frontier-src/*.tar.gz')
assert len(src) == 1, src
os.makedirs('/kaggle/working/xbrl-frontier', exist_ok=True)
subprocess.run(['tar', '-xzf', src[0], '-C', '/kaggle/working/xbrl-frontier'], check=True)
os.chdir('/kaggle/working/xbrl-frontier')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('vllm', 'llmcompressor', 'autoawq') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "vllm=={pins['vllm']}" "pyarrow==25.0.1" "pyyaml==6.0.3"
!python -c "import vllm, torch; print(vllm.__version__, torch.__version__, torch.cuda.get_device_name(0))"

## 1. Base-model throughput benchmark (fp16, FinNI token shape)

In [ ]:
!python -m serving.phase0_benchmark --out /kaggle/working/phase0_kaggle_t4

## 2. Quantize the base students for the smoke test (never scored)
INT8 W8A8 with llm-compressor in the main environment; AWQ-INT4 with AutoAWQ in its own virtualenv because of its transformers bound.

In [ ]:
!pip install -q "llmcompressor=={pins['llmcompressor']}"
!python -m serving.quantize_base --only int8 --out /kaggle/working/smoke_checkpoints

In [ ]:
!python -m venv /kaggle/working/awqenv
!/kaggle/working/awqenv/bin/pip install -q "autoawq=={pins['autoawq']}" "pyyaml==6.0.3" "pyarrow==25.0.1" huggingface_hub
!/kaggle/working/awqenv/bin/python -m serving.quantize_base --only awq --out /kaggle/working/smoke_checkpoints

## 3. vLLM smoke test of every planned T4 variant

In [ ]:
!python -m serving.smoke_test --card kaggle_t4 --checkpoints /kaggle/working/smoke_checkpoints --out /kaggle/working/smoke_kaggle_t4

In [ ]:
!tar -czf /kaggle/working/smoke_checkpoints.tar.gz -C /kaggle/working smoke_checkpoints
!ls -la /kaggle/working/*.tar.gz